In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sys
sys.path.append('/media/dll-1/SSD 4TB/code/LEEDNet/')  # Replace with real path
from EEGDataLoader import EEGDataset
from models.LEEDNetModel import LEEDNetModel
import config
from train_eval import eval, train

model_saving_name = "/media/dll-1/SSD 4TB/EEG Datasets/experiment/nouman_conv_2/model/weights/testparameters.pt"

epochs = 50
batch_size = 64
hidden_size = 20
output_size = 3  # Binary classification
number_of_eeg_channels = 1
eval_batch = 1
threshold = 0.495
active_branches = [0]

/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_dataset = EEGDataset(config.train_data_csv, number_of_eeg_channels=number_of_eeg_channels)
val_dataset = EEGDataset(config.val_data_csv, number_of_eeg_channels=number_of_eeg_channels)
test_dataset = EEGDataset(config.test_data_csv, number_of_eeg_channels=number_of_eeg_channels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=eval_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=eval_batch, shuffle=False)

In [8]:
input = train_dataset.__getitem__(0)
input_x1 = input[1].shape
input_x2 = input[2].shape
input_x3 = input[3].shape
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

print(input_x1,input_x2,input_x3)
input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

model = LEEDNetModel(input_channels_list=[input_channels_list[0]], output_size=3)
model1 = LEEDNetModel(input_channels_list=[input_channels_list[0], input_channels_list[1]], output_size=3)
model2 = LEEDNetModel(input_channels_list=input_channels_list, output_size=3)

main_output, aux_outputs = model([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output.shape) 

# main_output1 = model1(torch.randn(1, 1, 1, 400))
# print("Output shape:", main_output1.shape) 
active_branches = [0, 2]
main_output1, aux_outputs1 = model1([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output1.shape) 

active_branches = [0, 1, 2]
main_output2, aux_outputs2 = model2([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output2.shape) 

input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
optimizer1 = torch.optim.AdamW(model1.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer1, mode='min', patience=3, factor=0.5)
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', patience=3, factor=0.5)

model.to(device)
model2.to(device)
model1.to(device)


torch.Size([1, 1, 400]) torch.Size([1, 33, 14]) torch.Size([1, 30, 400])
Output shape: torch.Size([1, 3])
Output shape: torch.Size([1, 3])
Output shape: torch.Size([1, 3])


LEEDNetModel(
  (branches): ModuleList(
    (0): _EEGBackbone(
      (conv1_t): Sequential(
        (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
      )
      (conv2_t): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
      )
      (conv3_t): Sequential(
        (0): Conv2d(64, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
        (3): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): GELU(approximate='

In [16]:
model.eval()
# total trainable params
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {params:,}  (~{params*4/1e6:.2f} MB in FP32)")  # 4 bytes/param


model1.eval()
# total trainable params
params1 = sum(p.numel() for p in model1.parameters() if p.requires_grad)
print(f"Params: {params1:,}  (~{params1*4/1e6:.2f} MB in FP32)")  # 4 bytes/param


model2.eval()
# total trainable params
params2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Params: {params2:,}  (~{params2*4/1e6:.2f} MB in FP32)")  # 4 bytes/param

Params: 1,294,297  (~5.18 MB in FP32)
Params: 2,908,847  (~11.64 MB in FP32)
Params: 4,842,533  (~19.37 MB in FP32)
